# ForgeEdge — Event Discovery

Questo notebook illustra l'utilizzo del modulo **EventDiscovery**, il primo passo della pipeline FORGE.

Il modulo prende in input una tabella di KPI (indicatori tecnici su OHLCV) e restituisce una lista di **Event Candidates**: condizioni booleane sulle serie temporali che si attivano in modo statisticamente consistente nel tempo.

## Pipeline interna

```
Step 0 — TypeClassifier    → classifica le colonne (CONTINUOUS / BINARY / CATEGORICAL)
Step 1 — FeatureGenerator  → genera feature derivate (ratio, spread_pct, diffnorm, bb_pct_b, ...)
Step 2 — TransformLayer    → applica trasformate (identity, pctrank, zscore, delta)
Step 3 — EventGenerator    → converte le serie in eventi booleani (threshold + crossing)
Step 4 — ConsistencyGate   → filtra gli eventi per volume, copertura, concentrazione, frequenza
Step 5 — ANDComposer       → combina coppie/triple di eventi con AND logico + ri-applica il gate
```


## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "../src")   # per esecuzione da notebooks/

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from forgedge import EventDiscovery, DiscoveryConfig
from forgedge.event_discovery.models import GateParams, WalkForwardConfig, FoldResult, ValidationResult

## 1. Caricamento e preprocessing

Il dataset contiene candele orarie per due simboli (ADAUSDC, DOGEUSDC).  
Filtriamo su un singolo simbolo e lasciamo che EventDiscovery gestisca la conversione del timestamp.

> **Nota**: il modulo rileva automaticamente l'unità del timestamp numerico (s / ms / us / ns)  
> a partire dal valore mediano della colonna. Non è necessario convertire manualmente.

In [ ]:
# Sostituisci con il percorso al tuo file
DATA_PATH = "../data/test1h.xlsx"
SYMBOL = "ADAUSDC"

df = pd.read_excel(DATA_PATH)
df = df[df["symbol"] == SYMBOL].copy()
df = df.sort_values("open_time")

print(f"Simbolo : {SYMBOL}")
print(f"Righe   : {len(df):,}")
print(f"Colonne : {list(df.columns)}")
print(f"\nPrime righe:")
df.head(3)

## 2. Configurazione del pipeline

`DiscoveryConfig` raccoglie tutti i parametri:

| Parametro | Default | Descrizione |
|---|---|---|
| `timestamp_col` | `"open_dt"` | Nome della colonna timestamp (int, datetime o string) |
| `gate_params.min_act` | 50 | Attivazioni totali minime |
| `gate_params.min_months` | 8 | Mesi distinti minimi con almeno 1 attivazione |
| `gate_params.max_conc` | 0.40 | Max quota attivazioni in un singolo mese |
| `gate_params.min_tpm` | 2.0 | Attivazioni medie per mese minime |
| `max_and_components` | 2 | Cardinalità massima delle composizioni AND (2 o 3) |
| `scale_free_overrides` | None | Override manuali per il rilevamento scale-free |

In [ ]:
config = DiscoveryConfig(
    timestamp_col="open_time",
    gate_params=GateParams(
        min_act=50,
        min_months=8,
        max_conc=0.40,
        min_tpm=2.0,
    ),
    max_and_components=2,
)

## 3. Esecuzione del pipeline

In [ ]:
ed = EventDiscovery(df, config)
candidates = ed.run()

print(f"Candidati totali : {len(candidates)}")
print(f"  di cui singoli : {sum(1 for c in candidates if len(c.components) == 1)}")
print(f"  di cui AND     : {sum(1 for c in candidates if len(c.components) > 1)}")

## 4. Summary DataFrame

`ed.summary()` restituisce un DataFrame piatto con le metriche principali di ogni candidato.

In [ ]:
summary_df = ed.summary()
summary_df.head(5)

### Costruzione manuale del summary con metriche aggiuntive

Utile per aggiungere colonne custom (es. `arity`, `transform`, `source`) non presenti nel summary standard.

In [ ]:
records = []
for c in candidates:
    g = c.consistency_gate    # GateResult
    s = c.activation_stats    # ActivationStats (include zero_months)
    comp = c.components[0]    # primo EventComponent
    records.append({
        "expression":  c.expression,
        "arity":       len(c.components),
        "n_act":       g.n_activations,
        "n_months":    g.n_active_months,
        "zero_months": s.zero_months,
        "max_conc":    round(g.max_monthly_share, 4),
        "tpm":         round(g.mean_tpm, 2),
        "source":      comp.source_feature,
        "transform":   comp.transform,
        "event_type":  comp.event_type,
        "_candidate":  c,
    })

summary = pd.DataFrame(records)
singles = summary[summary["arity"] == 1].sort_values("max_conc")
ands    = summary[summary["arity"] > 1].sort_values(["max_conc", "n_act"], ascending=[True, False])

print(f"Singles: {len(singles)} | ANDs: {len(ands)}")

## 5. Migliori eventi singoli

Ordinati per **concentrazione mensile minima** (`max_conc`): valori bassi indicano  
attivazioni ben distribuite nel tempo — segnali più robusti e non legati a un singolo periodo.

In [ ]:
cols = ["expression", "n_act", "n_months", "zero_months", "max_conc", "tpm", "transform"]
singles.head(15)[cols]

### Distribuzione per tipo di trasformata

In [ ]:
singles.groupby("transform")["n_act"].agg(["count", "mean", "median"]).round(1)

## 6. Migliori composizioni AND

Coppie di eventi singoli combinate con AND logico che superano il ConsistencyGate.  
La composizione AND è più selettiva (meno attivazioni) ma cattura condizioni di mercato più specifiche.

In [ ]:
cols_and = ["expression", "n_act", "n_months", "zero_months", "max_conc", "tpm"]
ands.head(15)[cols_and]

## 7. Ispezione di un candidato

Ogni `EventCandidate` espone:
- `expression` — formula leggibile della condizione
- `components` — lista di `EventComponent` (uno per ogni sotto-condizione nell'AND)
- `consistency_gate` — `GateResult` con le metriche del filtro
- `activation_stats` — `ActivationStats` con `zero_months`
- `event_series` — serie booleana (0/1/NaN) con DatetimeIndex, resample-ready

In [ ]:
best = ands.iloc[0]["_candidate"]

print(f"Expression  : {best.expression}")
print(f"\nGate metrics:")
g = best.consistency_gate
print(f"  n_activations    : {g.n_activations}")
print(f"  n_active_months  : {g.n_active_months}")
print(f"  zero_months      : {best.activation_stats.zero_months}")
print(f"  max_monthly_share: {g.max_monthly_share:.4f}")
print(f"  mean_tpm         : {g.mean_tpm:.2f}")
print(f"\nComponents ({len(best.components)}):")
for i, comp in enumerate(best.components):
    print(f"  [{i}] transform={comp.transform:<20} expression={comp.expression}")

## 8. Breakdown mensile

`event_series` ha il DatetimeIndex già impostato → `.resample()` funziona direttamente.

In [ ]:
monthly = best.event_series.resample("ME").sum()
monthly.index = monthly.index.strftime("%Y-%m")

monthly.plot(
    kind="bar",
    figsize=(14, 4),
    color="steelblue",
    width=0.8,
    title=f"Attivazioni mensili — {best.expression}",
    ylabel="Attivazioni",
    xlabel="Mese",
)

In [ ]:
# Tabella mensile per i candidati con più bassa concentrazione
print("Mesi non-zero:")
print(monthly[monthly > 0].to_string())

## 9. Classificazioni delle colonne

`ed.get_classifications()` restituisce il dizionario prodotto da `TypeClassifier` (Step 0).  
Utile per verificare quali colonne sono state riconosciute come scale-free e quali escluse.

In [ ]:
clf_dict = ed.get_classifications()

clf_rows = []
for col, clf in sorted(clf_dict.items()):
    clf_rows.append({
        "column":      col,
        "type":        clf.col_type.name,
        "n_distinct":  clf.n_distinct,
        "scale_free":  clf.effective_scale_free if clf.col_type.name == "CONTINUOUS" else None,
    })

pd.DataFrame(clf_rows)

## 10. Feature table di debug

`ed.df` è il DataFrame completo post-pipeline con:
- **DatetimeIndex** impostato dal modulo
- Le colonne originali (OHLCV, indicatori)
- Tutte le **feature derivate** generate da `FeatureGenerator` (ratio, diffnorm, spread_pct, bb_pct_b…)

Utile per ispezionare le feature calcolate o tracciare grafici delle serie sottostanti.

In [ ]:
print(f"Shape : {ed.df.shape}")
print(f"Index : {type(ed.df.index).__name__}  {ed.df.index[0]} → {ed.df.index[-1]}")

derived = [c for c in ed.df.columns if any(
    c.startswith(p) for p in ("ratio_", "spread_", "diffnorm_", "bb_", "rng_")
)]
print(f"\nFeature derivate ({len(derived)}):")
for col in derived:
    print(f"  {col}")

In [ ]:
# Esempio: visualizza una feature derivata usata nei top candidati
import matplotlib.pyplot as plt

col_to_plot = "diffnorm_close_rsi14_rsi25"
if col_to_plot in ed.df.columns:
    fig, ax = plt.subplots(figsize=(14, 3))
    ed.df[col_to_plot].plot(ax=ax, lw=0.7, color="darkorange")
    ax.set_title(f"Feature derivata: {col_to_plot}")
    ax.set_ylabel("Valore")
    ax.axhline(0, color="gray", lw=0.5, linestyle="--")
    plt.tight_layout()

## 11. Confronto tra simboli (opzionale)

Confronta il numero e la qualità dei candidati tra ADAUSDC e DOGEUSDC.

In [ ]:
results = {}
df_all = pd.read_excel(DATA_PATH)

for sym in ["ADAUSDC", "DOGEUSDC"]:
    df_sym = df_all[df_all["symbol"] == sym].copy().sort_values("open_time")
    ed_sym = EventDiscovery(df_sym, config)
    cands  = ed_sym.run()
    results[sym] = {
        "total":   len(cands),
        "singles": sum(1 for c in cands if len(c.components) == 1),
        "ands":    sum(1 for c in cands if len(c.components) > 1),
        "best_max_conc": min(c.consistency_gate.max_monthly_share for c in cands) if cands else float("nan"),
    }

pd.DataFrame(results).T

---

## 12. Walk-Forward OOS Validation

La pipeline standard scopre gli eventi **su tutti i dati disponibili** (IS = 100%).  
Questo introduce un rischio: le soglie e i filtri del ConsistencyGate sono calibrati sull'intera serie storica, quindi non sappiamo se gli eventi reggono su dati mai visti.

**Walk-forward validation** risolve il problema in tre passi:

1. **Split temporale** — `train_ratio` riserva la prima frazione della serie come IS; il resto è OOS (mai toccato durante la discovery).
2. **Discovery IS-only** — tutta la pipeline (TypeClassifier → ANDComposer) gira solo sull'IS.
3. **Replay OOS** — ogni candidato IS viene riapplicato su `n_splits` finestre OOS consecutive via `apply()`. Per ogni finestra si esegue il ConsistencyGate con parametri auto-scalati alla lunghezza del fold. Un candidato è **OOS-stable** se supera almeno `min_pass_rate` delle finestre.

```
IS  ████████████████████████████  70 %
OOS                               ░░░░░░░░░░░░  30 %
        Fold 0      Fold 1      Fold 2
        ░░░░        ░░░░        ░░░░
```

### Parametri chiave

| Parametro | Default | Descrizione |
|---|---|---|
| `train_ratio` | `1.0` | Frazione IS (0 < x ≤ 1). Con `1.0` nessuno split. |
| `walk_forward.n_splits` | `3` | Numero di fold OOS uguali |
| `walk_forward.min_pass_rate` | `0.6` | Quota minima di fold superati per essere stable |
| `walk_forward.oos_gate_params` | `None` | Gate OOS custom; se `None` auto-scalato da IS |

### 12.1 Configurazione con split IS/OOS

In [ ]:
config_wf = DiscoveryConfig(
    timestamp_col="open_time",
    gate_params=GateParams(
        min_act=50,
        min_months=6,
        max_conc=0.40,
        min_tpm=2.0,
    ),
    max_and_components=2,
    train_ratio=0.70,                          # 70% IS, 30% OOS
    walk_forward=WalkForwardConfig(
        n_splits=3,                            # 3 fold OOS di uguale lunghezza
        min_pass_rate=0.60,                    # almeno 2/3 fold devono passare il gate
    ),
)

### 12.2 Esecuzione e periodi IS/OOS

In [ ]:
ed_wf = EventDiscovery(df, config_wf)
candidates_wf = ed_wf.run()

is_start, is_end   = ed_wf.is_period
oos_start, oos_end = ed_wf.oos_period

print(f"Periodo IS  : {is_start.date()} → {is_end.date()}")
print(f"Periodo OOS : {oos_start.date()} → {oos_end.date()}")
print()
print(f"Candidati IS totali  : {len(candidates_wf)}")
stable = ed_wf.validated_candidates()
print(f"Candidati OOS-stable : {len(stable)}  ({len(stable)/len(candidates_wf)*100:.1f}% di retention)")

### 12.3 Summary con colonne OOS

Quando la walk-forward è configurata, `summary()` aggiunge automaticamente quattro colonne:

| Colonna | Descrizione |
|---|---|
| `oos_pass_rate` | Quota di fold OOS superati (0.0–1.0) |
| `oos_n_passed` | Numero assoluto di fold superati |
| `oos_n_folds` | Totale fold valutati |
| `oos_stable` | `True` se `oos_pass_rate >= min_pass_rate` |

In [ ]:
summary_wf = ed_wf.summary()

oos_cols = ["expression", "n_activations", "n_active_months", "max_monthly_share", "mean_tpm",
            "oos_pass_rate", "oos_n_passed", "oos_n_folds", "oos_stable"]
(
    summary_wf[oos_cols]
    .sort_values(["oos_stable", "oos_pass_rate", "max_monthly_share"],
                 ascending=[False, False, True])
    .head(15)
)

### 12.4 Ispezione dei fold di un candidato OOS-stable

In [ ]:
if not stable:
    print("Nessun candidato OOS-stable con i parametri correnti. "
          "Prova ad abbassare min_pass_rate o min_act nel gate.")
else:
    best_stable = stable[0]
    v = best_stable.validation

    print(f"Expression  : {best_stable.expression}")
    print(f"OOS result  : {v.n_passed}/{v.n_folds} fold passati  (pass_rate={v.pass_rate:.0%})")
    print()
    print(f"{'Fold':<6} {'Rows':<8} {'n_act':<8} {'n_months':<10} {'max_conc':<10} {'tpm':<8} {'Passed'}")
    print("-" * 60)
    for fr in v.fold_results:
        g = fr.gate_result
        status = "✓" if fr.passed else "✗"
        print(f"{fr.fold_idx:<6} {fr.n_rows:<8} {g.n_activations:<8} "
              f"{g.n_active_months:<10} {g.max_monthly_share:<10.3f} "
              f"{g.mean_tpm:<8.2f} {status}")

### 12.5 Distribuzione pass_rate — tutti i candidati IS

In [ ]:
pass_rates = [c.validation.pass_rate for c in candidates_wf if c.validation is not None]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Istogramma pass_rate
axes[0].hist(pass_rates, bins=[0, 0.2, 0.4, 0.6, 0.8, 1.001],
             color="steelblue", edgecolor="white", rwidth=0.85)
axes[0].axvline(config_wf.walk_forward.min_pass_rate, color="red",
                linestyle="--", lw=1.5, label=f"min_pass_rate={config_wf.walk_forward.min_pass_rate}")
axes[0].set_title("Distribuzione OOS pass_rate")
axes[0].set_xlabel("pass_rate")
axes[0].set_ylabel("N candidati")
axes[0].legend()

# IS vs OOS stable per arity
arity_is  = [len(c.components) for c in candidates_wf]
arity_oos = [len(c.components) for c in stable]
for label, data, color in [("IS totali", arity_is, "steelblue"), ("OOS stable", arity_oos, "seagreen")]:
    counts = pd.Series(data).value_counts().sort_index()
    axes[1].bar(counts.index + (0.2 if label == "OOS stable" else -0.2),
                counts.values, width=0.38, label=label, color=color, alpha=0.85)
axes[1].set_title("Candidati per arity: IS vs OOS-stable")
axes[1].set_xlabel("N componenti (arity)")
axes[1].set_ylabel("N candidati")
axes[1].set_xticks([1, 2, 3])
axes[1].legend()

plt.tight_layout()

### 12.6 Gate params auto-scaling

Quando `oos_gate_params=None`, i parametri del ConsistencyGate vengono ridotti proporzionalmente alla lunghezza del fold OOS rispetto all'IS:

- `min_act` e `min_months` scalano con il rapporto `n_oos_bars / n_is_bars`
- `max_conc` e `min_tpm` rimangono invariati (sono rate/frazioni, non contatori assoluti)

Esempio: IS=6000 barre, fold OOS=600 barre → `min_act` scende di 10×.

In [ ]:
from forgedge.event_discovery.discovery import _scale_gate_params

n_is  = ed_wf._split_idx
n_oos_total = len(df) - n_is
fold_size = n_oos_total // config_wf.walk_forward.n_splits

scaled = _scale_gate_params(config_wf.gate_params, n_oos_bars=fold_size, n_is_bars=n_is)

print(f"IS bars       : {n_is}")
print(f"OOS total bars: {n_oos_total}  →  fold size: {fold_size}")
print()
print(f"{'Param':<14} {'IS (originale)':<20} {'OOS fold (scalato)'}")
print("-" * 50)
for attr in ("min_act", "min_months", "max_conc", "min_tpm"):
    orig  = getattr(config_wf.gate_params, attr)
    sc    = getattr(scaled, attr)
    print(f"{attr:<14} {str(orig):<20} {sc}")